# CSC 4792 Data Mining and Warehousing

## Kabwe Municipal Council Dataset

### Project Information

- **Course:** CSC 4792 Data Mining and Warehousing
- **Academic Year:** 2025/26
- **Council:** Kabwe Municipal Council
- **Country:** Zambia
- **Official Website:** https://www.kabwecouncil.gov.zm

### Objective

The objective of this project is to collect, extract, clean,
transform, and curate information associated with Kabwe Municipal
Council from its official digital sources.

The resulting datasets will be stored as pipe-delimited CSV files
and published on Kaggle.

## 1. Libraries

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import re
from urllib.parse import urljoin
import pymupdf

In [4]:
print("Libraries loaded successfully.")

Libraries loaded successfully.


## 2. Data Source

In [5]:
BASE_URL = "https://www.kabwecouncil.gov.zm"

print(BASE_URL)

https://www.kabwecouncil.gov.zm


In [6]:
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

response = requests.get(
    BASE_URL,
    timeout=30,
    verify=False
)

print("Status code:", response.status_code)

Status code: 200


In [7]:
soup = BeautifulSoup(response.text, "html.parser")

In [8]:
links = []

for link in soup.find_all("a", href=True):
    text = link.get_text(" ", strip=True)
    url = urljoin(BASE_URL, link["href"])

    links.append({
        "text": text,
        "url": url
    })

links_df = pd.DataFrame(links)

links_df.head(20)

,text,url
0,,https://www.kabwecouncil.gov.zm#
1,Home,https://www.kabwecouncil.gov.zm/
2,About,https://www.kabwecouncil.gov.zm#
3,About Us,https://www.kabwecouncil.gov.zm/?page_id=2601
4,Mandate,https://www.kabwecouncil.gov.zm/?page_id=169
5,Who we are,https://www.kabwecouncil.gov.zm/?page_id=118
6,Departments,https://www.kabwecouncil.gov.zm/?page_id=770
7,office of the the town clerk,https://www.kabwecouncil.gov.zm/?page_id=2634
8,Dept of Human Resource and Administration,https://www.kabwecouncil.gov.zm/?page_id=2637
9,Dept of Health,https://www.kabwecouncil.gov.zm/?page_id=2640


In [9]:
print("Number of links:", len(links_df))

Number of links: 122


In [10]:
keywords = [
    "cdf",
    "project",
    "budget",
    "financial",
    "report",
    "development",
    "idp",
    "ward",
    "revenue",
    "procurement",
    "minutes"
]

pattern = "|".join(keywords)

relevant_links = links_df[
    links_df["text"].str.contains(
        pattern,
        case=False,
        na=False
    )
]

relevant_links

,text,url
27,CDF,https://www.kabwecouncil.gov.zm/?page_id=2542
28,CDF GUIDLINES,https://www.kabwecouncil.gov.zm/?page_id=2579
29,CDF branding guidelines,https://www.kabwecouncil.gov.zm/?page_id=3674
66,CDF,https://www.kabwecouncil.gov.zm/?page_id=2542
67,CDF GUIDLINES,https://www.kabwecouncil.gov.zm/?page_id=2579
68,CDF branding guidelines,https://www.kabwecouncil.gov.zm/?page_id=3674
91,CDF Skills Bursaries Applicants,https://www.katetecouncil.gov.zm/wp-content/up...
108,CDF PROJECT MONITORING BY KABWE MUNICIPAL COUN...,https://www.kabwecouncil.gov.zm/?p=4386
115,Ministry of Local Government and Rural Develop...,https://www.mlgrd.gov.zm/


In [11]:
cdf_columns = [
    "project_id",
    "year",
    "constituency",
    "project_name",
    "project_description",
    "project_type",
    "ward",
    "project_site",
    "sector",
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount",
    "status",
    "source_url"
]

cdf_projects = pd.DataFrame(columns=cdf_columns)

cdf_projects

,project_id,year,constituency,project_name,project_description,project_type,ward,project_site,sector,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url


In [13]:
import requests

pdf_url = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2024/11/"
    "2024-Bwacha-community-projects-Recieved.pdf"
)

pdf_response = requests.get(
    pdf_url,
    timeout=60,
    verify=False
)

print("Status code:", pdf_response.status_code)
print("Content-Type:", pdf_response.headers.get("Content-Type"))
print("File size:", len(pdf_response.content), "bytes")
print("First 20 bytes:", pdf_response.content[:20])

Status code: 200
Content-Type: application/pdf
File size: 93259 bytes
First 20 bytes: b'%PDF-1.5\r\n%\xb5\xb5\xb5\xb5\r\n1 0'


In [14]:
pdf_path = "../data/raw/2024_bwacha_cdf_projects.pdf"

with open(pdf_path, "wb") as file:
    file.write(pdf_response.content)

print("PDF downloaded successfully.")

PDF downloaded successfully.


In [16]:
import requests
import pdfplumber
import pandas as pd

pdf_url = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2024/11/"
    "2024-Bwacha-community-projects-Recieved.pdf"
)

response = requests.get(
    pdf_url,
    timeout=60,
    verify=False
)

print("Status code:", response.status_code)
print("File size:", len(response.content), "bytes")
print("Content-Type:", response.headers.get("Content-Type"))
print("First 20 bytes:", response.content[:20])

Status code: 200
File size: 93259 bytes
Content-Type: application/pdf
First 20 bytes: b'%PDF-1.5\r\n%\xb5\xb5\xb5\xb5\r\n1 0'


In [17]:
pdf_path = "../data/raw/2024_bwacha_cdf_projects.pdf"

with open(pdf_path, "wb") as file:
    file.write(response.content)

print("PDF saved to:", pdf_path)

PDF saved to: ../data/raw/2024_bwacha_cdf_projects.pdf


In [18]:
with pdfplumber.open(pdf_path) as pdf:
    print("Number of pages:", len(pdf.pages))

    for page_number, page in enumerate(pdf.pages):
        tables = page.extract_tables()

        print(
            f"Page {page_number + 1}: "
            f"{len(tables)} table(s) found"
        )

Number of pages: 5
Page 1: 1 table(s) found
Page 2: 1 table(s) found
Page 3: 1 table(s) found
Page 4: 1 table(s) found
Page 5: 1 table(s) found


In [19]:
with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[0]
    tables = page.extract_tables()

    table = tables[0]

    for row in table[:10]:
        print(row)

['2024 CDF COMMUNITY PROJECTS SUBMISSION - BWACHA CONSTITUENCY\nKABWE MUNICIPAL COUNCIL', None, None, None, None, None, None, None, None, None]
['No.', 'Project Name', 'Project Description', 'Ward', 'Project\nSite/Location', 'Application\nAmount', "Engineers'\nEstimates", 'Approved\nAmount', 'Contract\nAmount', 'Status']
['Education', None, None, None, None, None, None, None, None, None]
['1', 'Construction of 1X3 Classroom\nBlock and Teachers Houses in\nKangomba ward', 'Construction of 1X3\nClassroom Block and\nTeachers Houses in\nKangomba ward - Kalima zone', 'Kangomba', 'Kangomba\nward', '', '', '', '', '']
['2', 'Construction of 1X4 Classroom\nBlock at Kagomba Primary\nSchool', 'Construction of 1X4\nClassroom Block at Kagomba\nPrimary School', 'Kangomba', 'Kangomba\nPrimary\nSchool', '', '', '', '', '']
['3', 'Construction of 1X4 Classroom\nBlock at Kagomba Primary\nSchool', 'Construction of 1X4\nClassroom Block at Kagomba\nPrimary School', 'Kangomba', 'Kangomba\nPrimary\nSchool', 

In [20]:
df = pd.DataFrame(
    table[1:],
    columns=table[0]
)

df.head()

,2024 CDF COMMUNITY PROJECTS SUBMISSION - BWACHA CONSTITUENCY\nKABWE MUNICIPAL COUNCIL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,No.,Project Name,Project Description,Ward,Project\nSite/Location,Application\nAmount,Engineers'\nEstimates,Approved\nAmount,Contract\nAmount,Status
1,Education,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,Construction of 1X3 Classroom\nBlock and Teach...,Construction of 1X3\nClassroom Block and\nTeac...,Kangomba,Kangomba\nward,,,,,
3,2,Construction of 1X4 Classroom\nBlock at Kagomb...,Construction of 1X4\nClassroom Block at Kagomb...,Kangomba,Kangomba\nPrimary\nSchool,,,,,
4,3,Construction of 1X4 Classroom\nBlock at Kagomb...,Construction of 1X4\nClassroom Block at Kagomb...,Kangomba,Kangomba\nPrimary\nSchool,,,,,


In [21]:
print(df.shape)
print(df.columns.tolist())

(8, 10)
['2024 CDF COMMUNITY PROJECTS SUBMISSION - BWACHA CONSTITUENCY\nKABWE MUNICIPAL COUNCIL', nan, nan, nan, nan, nan, nan, nan, nan, nan]


In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 10 columns):
 #   Column                                                                                Non-Null Count  Dtype
---  ------                                                                                --------------  -----
 0   2024 CDF COMMUNITY PROJECTS SUBMISSION - BWACHA CONSTITUENCY
KABWE MUNICIPAL COUNCIL  8 non-null      str  
 1   nan                                                                                   7 non-null      str  
 2   nan                                                                                   7 non-null      str  
 3   nan                                                                                   7 non-null      str  
 4   nan                                                                                   7 non-null      str  
 5   nan                                                                                   7 non-null      str  
 6   n

In [23]:
for row in table[:10]:
    print(row)

['2024 CDF COMMUNITY PROJECTS SUBMISSION - BWACHA CONSTITUENCY\nKABWE MUNICIPAL COUNCIL', None, None, None, None, None, None, None, None, None]
['No.', 'Project Name', 'Project Description', 'Ward', 'Project\nSite/Location', 'Application\nAmount', "Engineers'\nEstimates", 'Approved\nAmount', 'Contract\nAmount', 'Status']
['Education', None, None, None, None, None, None, None, None, None]
['1', 'Construction of 1X3 Classroom\nBlock and Teachers Houses in\nKangomba ward', 'Construction of 1X3\nClassroom Block and\nTeachers Houses in\nKangomba ward - Kalima zone', 'Kangomba', 'Kangomba\nward', '', '', '', '', '']
['2', 'Construction of 1X4 Classroom\nBlock at Kagomba Primary\nSchool', 'Construction of 1X4\nClassroom Block at Kagomba\nPrimary School', 'Kangomba', 'Kangomba\nPrimary\nSchool', '', '', '', '', '']
['3', 'Construction of 1X4 Classroom\nBlock at Kagomba Primary\nSchool', 'Construction of 1X4\nClassroom Block at Kagomba\nPrimary School', 'Kangomba', 'Kangomba\nPrimary\nSchool', 

In [24]:
all_rows = []

with pdfplumber.open(pdf_path) as pdf:
    for page_number, page in enumerate(pdf.pages, start=1):
        tables = page.extract_tables()

        for table in tables:
            for row in table:
                if row:
                    all_rows.append(row)

print("Total rows extracted:", len(all_rows))

Total rows extracted: 47


In [25]:
project_rows = []

for row in all_rows:
    if row[0] is not None and str(row[0]).strip().isdigit():
        project_rows.append(row)

print("Number of project rows:", len(project_rows))

Number of project rows: 36


In [26]:
for row in project_rows[:5]:
    print(row)

['1', 'Construction of 1X3 Classroom\nBlock and Teachers Houses in\nKangomba ward', 'Construction of 1X3\nClassroom Block and\nTeachers Houses in\nKangomba ward - Kalima zone', 'Kangomba', 'Kangomba\nward', '', '', '', '', '']
['2', 'Construction of 1X4 Classroom\nBlock at Kagomba Primary\nSchool', 'Construction of 1X4\nClassroom Block at Kagomba\nPrimary School', 'Kangomba', 'Kangomba\nPrimary\nSchool', '', '', '', '', '']
['3', 'Construction of 1X4 Classroom\nBlock at Kagomba Primary\nSchool', 'Construction of 1X4\nClassroom Block at Kagomba\nPrimary School', 'Kangomba', 'Kangomba\nPrimary\nSchool', '', '', '', '', '']
['4', 'Construction of 1X3 Classroom\nBlock at Mine Primary School', 'Construction of 1X3\nClassroom Block at Mine\nPrimary School -\nMutwewansofu zone', 'Kangomba', 'Mine Primary\nSchool', '', '', '', '', '']
['5', 'Repairing of the Mono-Pump,\nConstruction of Toilets for Pre-\nSchool Pupils, Procurement of\nDesks at Mary Chidgey\nCommunity Primary School', 'Repairing

In [27]:
columns = [
    "project_number",
    "project_name",
    "project_description",
    "ward",
    "project_site",
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount",
    "status"
]

df = pd.DataFrame(project_rows, columns=columns)

df.head()

,project_number,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of 1X3 Classroom\nBlock and Teach...,Construction of 1X3\nClassroom Block and\nTeac...,Kangomba,Kangomba\nward,,,,,
1,2,Construction of 1X4 Classroom\nBlock at Kagomb...,Construction of 1X4\nClassroom Block at Kagomb...,Kangomba,Kangomba\nPrimary\nSchool,,,,,
2,3,Construction of 1X4 Classroom\nBlock at Kagomb...,Construction of 1X4\nClassroom Block at Kagomb...,Kangomba,Kangomba\nPrimary\nSchool,,,,,
3,4,Construction of 1X3 Classroom\nBlock at Mine P...,Construction of 1X3\nClassroom Block at Mine\n...,Kangomba,Mine Primary\nSchool,,,,,
4,5,"Repairing of the Mono-Pump,\nConstruction of T...","Repairing of the Mono-Pump,\nConstruction of T...",Kangomba,Mary Chidgey\nCommunity\nSchool,,,,,


In [28]:
print(df.shape)
print(df.columns.tolist())

(36, 10)
['project_number', 'project_name', 'project_description', 'ward', 'project_site', 'application_amount', 'engineers_estimate', 'approved_amount', 'contract_amount', 'status']


In [29]:
for column in df.columns:
    df[column] = (
        df[column]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

df.head()

,project_number,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of 1X3 Classroom Block and Teache...,Construction of 1X3 Classroom Block and Teache...,Kangomba,Kangomba ward,,,,,
1,2,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,Kangomba,Kangomba Primary School,,,,,
2,3,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,Kangomba,Kangomba Primary School,,,,,
3,4,Construction of 1X3 Classroom Block at Mine Pr...,Construction of 1X3 Classroom Block at Mine Pr...,Kangomba,Mine Primary School,,,,,
4,5,"Repairing of the Mono-Pump, Construction of To...","Repairing of the Mono-Pump, Construction of To...",Kangomba,Mary Chidgey Community School,,,,,


In [30]:
print(df.loc[0, "project_name"])
print(df.loc[0, "project_description"])

Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward
Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward - Kalima zone


In [31]:
df.insert(1, "year", 2024)
df.insert(2, "constituency", "Bwacha")

In [32]:
df.head()

,project_number,year,constituency,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,2024,Bwacha,Construction of 1X3 Classroom Block and Teache...,Construction of 1X3 Classroom Block and Teache...,Kangomba,Kangomba ward,,,,,
1,2,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,Kangomba,Kangomba Primary School,,,,,
2,3,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,Kangomba,Kangomba Primary School,,,,,
3,4,2024,Bwacha,Construction of 1X3 Classroom Block at Mine Pr...,Construction of 1X3 Classroom Block at Mine Pr...,Kangomba,Mine Primary School,,,,,
4,5,2024,Bwacha,"Repairing of the Mono-Pump, Construction of To...","Repairing of the Mono-Pump, Construction of To...",Kangomba,Mary Chidgey Community School,,,,,


In [33]:
df["source_url"] = pdf_url
df["source_document"] = "2024 Bwacha CDF Community Projects Submission"

In [34]:
pd.set_option("display.max_colwidth", 100)

df.head(10)

,project_number,year,constituency,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
0,1,2024,Bwacha,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward - Kalima zone,Kangomba,Kangomba ward,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
1,2,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
2,3,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
3,4,2024,Bwacha,Construction of 1X3 Classroom Block at Mine Primary School,Construction of 1X3 Classroom Block at Mine Primary School - Mutwewansofu zone,Kangomba,Mine Primary School,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
4,5,2024,Bwacha,"Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...","Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...",Kangomba,Mary Chidgey Community School,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
5,6,2024,Bwacha,Construction of a Primary School in Kawama ward,Construction of a Primary School in Kawama ward,Kawama,Kawama ward,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
6,7,2024,Bwacha,Construction of 10 Teachers Houses at Chitakata Community School,Construction of 10 Teachers Houses at Chitakata Community School,Muwowo East,Chitakata Commnuity School,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
7,8,2024,Bwacha,Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary School,Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary School,Muwowo East,Mukobeko Correctional Day Secondary School,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
8,9,2024,Bwacha,Construction of Youth Resource Centre,Construction of Youth Resource Centre,Bwacha,Bwacha ward,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
9,10,2024,Bwacha,Construction of a School Hall at Rapheal Kombe Secondary School,Construction of a School Hall at Rapheal Kombe Secondary School,Chimaniman i,Rapheal Kombe Secondary School,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission


In [35]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   project_number       36 non-null     str  
 1   year                 36 non-null     int64
 2   constituency         36 non-null     str  
 3   project_name         36 non-null     str  
 4   project_description  36 non-null     str  
 5   ward                 36 non-null     str  
 6   project_site         36 non-null     str  
 7   application_amount   36 non-null     str  
 8   engineers_estimate   36 non-null     str  
 9   approved_amount      36 non-null     str  
 10  contract_amount      36 non-null     str  
 11  status               36 non-null     str  
 12  source_url           36 non-null     str  
 13  source_document      36 non-null     str  
dtypes: int64(1), str(13)
memory usage: 4.1 KB


In [36]:
df["ward"].value_counts()

ward
Kangomba        8
Kawama          8
Chimaniman i    6
Bwacha          4
Muwowo East     2
Munyama         2
Chililalila     2
Chinyama        2
Ngungu          2
Name: count, dtype: int64

In [37]:
df["status"].value_counts(dropna=False)

status
    36
Name: count, dtype: int64

In [38]:
financial_columns = [
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount"
]

for column in financial_columns:
    print(f"\n--- {column} ---")
    print(df[column].unique()[:20])


--- application_amount ---
<StringArray>
['']
Length: 1, dtype: str

--- engineers_estimate ---
<StringArray>
['']
Length: 1, dtype: str

--- approved_amount ---
<StringArray>
['']
Length: 1, dtype: str

--- contract_amount ---
<StringArray>
['']
Length: 1, dtype: str


In [39]:
def clean_amount(value):
    value = str(value).strip()

    # Treat blank values as missing
    if value == "" or value.lower() in ["nan", "none", "n/a", "na", "-"]:
        return pd.NA

    # Remove currency symbols, commas and other non-numeric characters
    value = re.sub(r"[^0-9.\-]", "", value)

    if value == "":
        return pd.NA

    return float(value)

In [40]:
for column in financial_columns:
    df[column] = df[column].apply(clean_amount)

In [41]:
df[financial_columns].head(10)

,application_amount,engineers_estimate,approved_amount,contract_amount
0,<NA>,<NA>,<NA>,<NA>
1,<NA>,<NA>,<NA>,<NA>
2,<NA>,<NA>,<NA>,<NA>
3,<NA>,<NA>,<NA>,<NA>
4,<NA>,<NA>,<NA>,<NA>
5,<NA>,<NA>,<NA>,<NA>
6,<NA>,<NA>,<NA>,<NA>
7,<NA>,<NA>,<NA>,<NA>
8,<NA>,<NA>,<NA>,<NA>
9,<NA>,<NA>,<NA>,<NA>


In [42]:
df[financial_columns].head(10)

,application_amount,engineers_estimate,approved_amount,contract_amount
0,<NA>,<NA>,<NA>,<NA>
1,<NA>,<NA>,<NA>,<NA>
2,<NA>,<NA>,<NA>,<NA>
3,<NA>,<NA>,<NA>,<NA>
4,<NA>,<NA>,<NA>,<NA>
5,<NA>,<NA>,<NA>,<NA>
6,<NA>,<NA>,<NA>,<NA>
7,<NA>,<NA>,<NA>,<NA>
8,<NA>,<NA>,<NA>,<NA>
9,<NA>,<NA>,<NA>,<NA>


In [43]:
df[financial_columns].dtypes

application_amount    object
engineers_estimate    object
approved_amount       object
contract_amount       object
dtype: object

In [44]:
df[financial_columns].isna().sum()

application_amount    36
engineers_estimate    36
approved_amount       36
contract_amount       36
dtype: int64

In [45]:
text_columns = [
    "project_name",
    "project_description",
    "ward",
    "project_site",
    "status"
]

for column in text_columns:
    df[column] = (
        df[column]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

In [46]:
df.head(10)

,project_number,year,constituency,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
0,1,2024,Bwacha,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward - Kalima zone,Kangomba,Kangomba ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
1,2,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
2,3,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
3,4,2024,Bwacha,Construction of 1X3 Classroom Block at Mine Primary School,Construction of 1X3 Classroom Block at Mine Primary School - Mutwewansofu zone,Kangomba,Mine Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
4,5,2024,Bwacha,"Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...","Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...",Kangomba,Mary Chidgey Community School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
5,6,2024,Bwacha,Construction of a Primary School in Kawama ward,Construction of a Primary School in Kawama ward,Kawama,Kawama ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
6,7,2024,Bwacha,Construction of 10 Teachers Houses at Chitakata Community School,Construction of 10 Teachers Houses at Chitakata Community School,Muwowo East,Chitakata Commnuity School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
7,8,2024,Bwacha,Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary School,Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary School,Muwowo East,Mukobeko Correctional Day Secondary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
8,9,2024,Bwacha,Construction of Youth Resource Centre,Construction of Youth Resource Centre,Bwacha,Bwacha ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
9,10,2024,Bwacha,Construction of a School Hall at Rapheal Kombe Secondary School,Construction of a School Hall at Rapheal Kombe Secondary School,Chimaniman i,Rapheal Kombe Secondary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission


In [47]:
duplicates = df.duplicated(
    subset=[
        "project_name",
        "ward",
        "project_site"
    ]
)

print("Duplicate rows:", duplicates.sum())

Duplicate rows: 1


In [48]:
print(df["project_number"].unique())

<StringArray>
['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12']
Length: 12, dtype: str


In [49]:
print("Total projects:", len(df))

Total projects: 36


In [50]:
processed_path = "../data/processed/kabwe_2024_bwacha_cdf_projects.csv"

df.to_csv(
    processed_path,
    sep="|",
    index=False
)

print("Saved:", processed_path)

Saved: ../data/processed/kabwe_2024_bwacha_cdf_projects.csv


In [51]:
test_df = pd.read_csv(
    processed_path,
    sep="|"
)

print("Rows:", len(test_df))
print("Columns:", len(test_df.columns))

test_df.head()

Rows: 36
Columns: 14


,project_number,year,constituency,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
0,1,2024,Bwacha,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward - Kalima zone,Kangomba,Kangomba ward,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
1,2,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
2,3,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
3,4,2024,Bwacha,Construction of 1X3 Classroom Block at Mine Primary School,Construction of 1X3 Classroom Block at Mine Primary School - Mutwewansofu zone,Kangomba,Mine Primary School,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
4,5,2024,Bwacha,"Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...","Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...",Kangomba,Mary Chidgey Community School,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
